# CloudTrend 통합 장기 데이터 수집기 — Toss Open API + KRX

이 노트북은 CloudTrend 장기 백테스트/스크리닝 입력자료를 **원천 출처별로 분리해서** 수집합니다.

### 고속·안전 실행 방식 (v2)
- Toss I/O는 8개 worker로 겹쳐 처리하되 공식 `Rate Limits Group`별 간격을 공동 적용합니다.
- KRX/pykrx 호출은 전체 worker에서 한 번에 1개만 실행합니다.
- 작은 단계별 캐시는 Colab 로컬 SSD에, 완료 종목은 10개 단위로 Google Drive에 저장합니다.
- 중단 시 남은 완료 종목도 즉시 저장하며, 재실행하면 완료 종목과 기존 v1 캐시를 자동 재사용합니다.
- 연결 5초·응답 15초 timeout, 재시도/429 횟수와 실제 HTTP 요청 수를 진행률에 표시합니다.

### 데이터 원칙
- **가격(수정 OHLCV): Toss** `adjusted=true`
- **정확 거래대금·과거 시가총액·상장주식수: KRX**
- **종목별 실제 순매수금액(KRW): KRX**
- **개인/외국인/기관/기타법인 순매수수량 + 기관 7개 세부주체 + 외국인 보유량/보유율: Toss**
- **공매도·대차·신용잔고·프로그램매매: Toss**
- **ETF NAV·괴리율·추적오차·순자산총액(AUM)·상장좌수·기초지수: KRX**
- KRX 값이 없을 때 `marketCap`, `foreignNetBuyValue`, `institutionNetBuyValue`를 임의 추정하지 않습니다.
- `tradingValue`만 CloudTrend 호환성을 위해 KRX 값이 없을 때 `close × volume` 프록시를 사용하며, `tradingValueSource`에 명시합니다.
- Toss 종목별 투자자 수량은 KRX+NXT 통합 기준일 수 있고 KRX 순매수금액은 KRX 기준이므로 서로 환산하지 않습니다.

### 환경변수
- `TOSS_CLIENT_ID`
- `TOSS_CLIENT_SECRET`
- `KRX_ID`
- `KRX_PW`

### 포함 범위
1. KRX 시가총액·거래대금·상장주식수
2. KRX 외국인/기관/개인/기타법인 및 기관 세부 순매수금액
3. Toss 최신 investor-trading 전체 주요 필드
4. Toss 공매도·대차·신용거래
5. Toss 프로그램매매
6. KRX ETF NAV/AUM/상장좌수/괴리율/추적오차

> 1,250거래일·전체 수급을 켜면 종목당 Toss 호출은 최대 약 72회입니다. 첫 실행은 여전히 장시간 작업이지만, 기존 직렬 실행의 네트워크 대기를 병렬로 겹쳐 처리합니다.


In [ ]:
# 설치: 최신 pykrx와 기본 의존성
%pip install -q --upgrade pykrx requests pandas tqdm

import os
import re
import sys
import time
import random
import json
import warnings
import threading
import math
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display, FileLink

from pykrx import stock
from pykrx.website.krx.etx.core import 개별종목시세_ETF
from pykrx.website.krx.etx.ticker import get_etx_isin
from pykrx.website.krx.etx import wrap as etx_wrap

pd.set_option("display.max_columns", 200)
warnings.filterwarnings("default")

# Python 3.13 + jupyter_client 자체의 utcnow 경고가 tqdm 화면을 반복해서 밀어내는
# 현상만 숨깁니다. 수집 코드에서 발생하는 다른 경고는 그대로 표시합니다.
warnings.filterwarnings(
    "ignore",
    message=r"datetime\.datetime\.utcnow\(\) is deprecated.*",
    category=DeprecationWarning,
    module=r"jupyter_client\.session",
)

# Colab이면 Google Drive를 한 번 마운트합니다. 완료 체크포인트만 Drive에 쓰고,
# API 단계 캐시는 빠른 /content 로컬 디스크를 사용합니다.
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)


In [ ]:
# ============================================================
# 1. 실행 설정 / CloudTrend universe
# ============================================================

BASE = "https://openapi.tossinvest.com"
OPENAPI_SPEC_URL = "https://openapi.tossinvest.com/openapi-docs/latest/openapi.json"
SCHEMA_REVIEW_DATE = "2026-09-12"

ENV_KEYS = ["TOSS_CLIENT_ID", "TOSS_CLIENT_SECRET", "KRX_ID", "KRX_PW"]
missing_env = [key for key in ENV_KEYS if not os.environ.get(key)]
if missing_env:
    raise EnvironmentError(
        "필수 환경변수가 없습니다: " + ", ".join(missing_env)
    )

CLIENT_ID = os.environ["TOSS_CLIENT_ID"]
CLIENT_SECRET = os.environ["TOSS_CLIENT_SECRET"]

# 장기 백테스트 기준. 1,250 거래일 ~= 약 5년
TARGET_COUNT = 1250
FLOW_COUNT = 1250

# 출력 분할 크기
CHUNK_DAYS = 158

# 실행 옵션 — 요청하신 1~6번을 모두 기본 활성화
COLLECT_KRX_MARKET = True
COLLECT_KRX_INVESTOR_VALUE = True
COLLECT_TOSS_INVESTOR = True
COLLECT_TOSS_PROGRAM = True
COLLECT_TOSS_SHORT = True
COLLECT_TOSS_CREDIT = True
COLLECT_TOSS_LENDING = True
COLLECT_KRX_ETF = True

# ------------------------------------------------------------
# 성능 / 체크포인트 설정
# ------------------------------------------------------------
# Toss HTTP 대기시간을 겹쳐 처리하는 작업 수입니다. 공식 Rate Limits Group별
# 속도 제한은 아래 gap으로 별도 적용되므로 worker 수가 곧 초당 요청 수는 아닙니다.
TOSS_WORKERS = 8

# 공식 OpenAPI의 그룹 분리를 그대로 사용합니다.
# STOCK_TRADING_TREND는 5개 수급 endpoint가 같은 버킷을 공유합니다.
TOSS_CHART_GAP = 0.15          # MARKET_DATA_CHART: 약 6.7 req/s
TOSS_TREND_GAP = 0.12          # STOCK_TRADING_TREND: 약 8.3 req/s
TOSS_STOCK_GAP = 0.35          # STOCK: 종목 기본정보
TOSS_INDICATOR_GAP = 0.12      # MARKET_INDICATOR: 지수 수급
TOSS_CONNECT_TIMEOUT = 5
TOSS_READ_TIMEOUT = 15
TOSS_MAX_RETRIES = 5

# KRX/pykrx는 로그인 세션과 서버 부하를 고려해 전체 호출을 1개씩 직렬화합니다.
# Toss worker가 KRX 호출을 기다리는 동안 다른 worker의 Toss I/O는 계속 진행됩니다.
KRX_MIN_GAP = 0.35

# API 단계 캐시는 Colab 로컬 SSD에 저장해 Drive FUSE의 작은 파일 쓰기 병목을 피합니다.
LOCAL_WORK_ROOT = (
    Path("/content/cloudtrend_toss_krx_work")
    if IN_COLAB else Path(".cloudtrend_toss_krx_work")
)
CACHE_ROOT = LOCAL_WORK_ROOT / "stage_cache_v2"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

# 완료 종목은 10개 단위 shard로 Drive에 기록합니다. 중단 시 남은 1~9개도 flush합니다.
DEFAULT_STATE_ROOT = (
    Path("/content/drive/MyDrive/CloudTrend/toss_krx_integrated")
    if IN_COLAB else Path("cloudtrend_toss_krx_integrated")
)
STATE_ROOT = Path(os.environ.get("CLOUDTREND_STATE_ROOT", str(DEFAULT_STATE_ROOT)))
CHECKPOINT_ROOT = STATE_ROOT / "checkpoints_v2"
OUTPUT_ROOT = STATE_ROOT / "outputs"
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_EVERY = 10

# 기존 v1 단계 캐시 및 종목별 CSV를 자동 탐색하여 앞서 완료한 24개도 재사용합니다.
AUTO_DISCOVER_LEGACY = True
LEGACY_CACHE_ROOTS = []
LEGACY_PARTIAL_FILES = {}
REFRESH_CACHE = False
FAIL_SYMBOL_ON_STAGE_ERROR = True
COLLECTION_STOP_EVENT = threading.Event()

STOCKS = ['000660', '005930', '009150', '402340', '005380', '066570', '035420', '034020', '042700', '011070', '006400',
 '010120', '042660', '036930', '028260', '196170', '475150', '012330', '012450', '105560', '403870', '034730',
 '080220', '298040', '017670', '240810', '032830', '329180', '267260', '079550', '086520', '000270', '222800',
 '373220', '005490', '068270', '006800', '278470', '055550', '096770', '006340', '010140', '047040', '000500',
 '007660', '000720', '000150', '010060', '095610', '002990', '108490', '010170', '319660', '277810', '353200',
 '086790', '018260', '207940', '058470', '058610', '073240', '015760', '064350', '347850', '062040', '028050',
 '003230', '214450', '247540', '009830', '128940', '001820', '316140', '000990', '307950', '009540', '089970',
 '047810', '000810', '028300', '003490', '006260', '003670', '006360', '066970', '039030', '035720', '047050',
 '064400', '067310', '272210', '352820', '001440', '161890', '051910', '087010', '257720', '016360', '010950',
 '089030', '141080', '093370', '000250', '336260', '119850', '082740', '483650', '071050', '004170', '950160',
 '226950', '267270', '011790', '178320', '033780', '095340', '078930', '440110', '090430', '052690', '086280',
 '298380', '103590', '043260', '011200', '322000', '003550', '034220', '204320', '005290', '090360', '001210',
 '039490', '131970', '084370', '037710', '067290', '454910', '131290', '007810', '010130', '032820', '098460',
 '267250', '004310', '005830', '031980', '138040', '319400', '009420', '281820', '079650', '090460', '083650',
 '192820', '290650', '024060', '024110', '124500', '356860', '347700', '051900', '001740', '375500', '018880',
 '460930', '000100', '218410', '259960', '443060', '100090', '069960', '466100', '078600', '001450', '357780',
 '036570', '399720', '237690', '069540', '023530', '058730', '024840', '008930', '323280', '476060', '004020',
 '161390', '332570', '489790', '323410', '030200', '088350', '000880', '030530', '420770', '005940', '033100',
 '112610', '439090', '122640', '376900', '108860', '183300', '017900', '022100', '263750', '064760', '085620',
 '071970', '377300', '001040', '413630', '100790', '310210', '138930', '180640', '007390', '003280', '126640',
 '021240', '241710', '326030', '082920', '397030', '295310', '032640', '019210', '008770', '320000', '003160',
 '083450', '044490', '002380', '181710', '028670', '033160', '097950', '090710', '271560', '437730', '088980',
 '017800', '101490', '145020', '006110', '041190', '457190', '074600', '035900', '025980', '356680', '004370',
 '195870', '126340', '175330', '120110', '004710', '103140', '166090', '096530', '064260', '214150', '064290',
 '011780', '450080', '241560', '014680', '037070', '007340', '170920', '020150', '125490', '139480', '001510',
 '017960', '458870', '475830', '304100', '006660', '036540', '027360', '140860', '232140', '200710', '091590',
 '086450', '327260', '417840', '475430', '126730', '031330', '011170', '005090', '348370', '144960', '282330',
 '100840', '077970', '448900', '079900', '361610', '065350', '198440', '101730', '419050', '388050', '457370',
 '003350', '456160', '046970', '034230', '003680', '051600', '041510', '139130', '161580', '229640', '445680',
 '052710', '033640', '330860', '019170', '041830', '059090', '252990', '011210', '005850', '010690', '249420',
 '005950', '189300', '001430', '317400', '039200', '234340', '204270', '298020', '112040', '032350', '006910',
 '030000', '000890', '097230', '005690', '441270', '089890', '064550', '005180', '035250', '049080', '044340',
 '226340', '099320', '263800', '036460', '050890', '122350', '025320', '045100', '456040', '123330', '417200',
 '033790', '046890', '199430', '251270', '127120', '032940', '214370', '264850', '298000', '294570', '195940',
 '047770', '010960', '011230', '005070', '029780', '140410', '006220', '383220', '077360', '095500', '003530',
 '014620', '396300', '270660', '060370', '026940', '161000', '465770', '059120', '025560', '047920', '137400',
 '463020', '023160', '330350', '032500', '189330', '111770', '094170', '081660', '290550', '348340', '228340',
 '358570', '290690', '089860', '060250', '213420', '328130', '092790', '048410', '122870', '138080', '107640',
 '092870', '001120', '455900', '065170', '004490', '102710', '066430', '160980', '038500', '439260', '484810',
 '389260', '383310', '049070', '004990', '171090', '425420', '115180', '012750', '200470', '009520', '006280',
 '499790', '036810', '007070', '003380', '102940', '032580', '069620', '010820', '003690', '004800', '251970',
 '114190', '023410', '013580', '039440', '058430', '108320', '005360', '459510', '042520', '000210', '005440',
 '114810', '075580', '003010', '302440', '039980', '078350', '484870', '036200', '033240', '101160', '068760',
 '293490', '445090', '005880', '281740', '078340', '067080', '006730', '009450', '456010', '007610', '089010',
 '222080', '000120', '468530', '219130', '174900', '004980', '094360', '294870', '382900', '348210', '053260',
 '002020', '053690', '462860', '204620', '080580', '462870', '011930', '006650', '285130', '130660', '037440',
 '425040', '009900', '009470', '018290', '045390', '056080', '253590', '237880', '419530', '002700', '002140',
 '388720', '121600', '280360', '000370', '187660', '018670', '001060', '085660', '086390', '192080', '014910',
 '092200', '482630', '005860', '389500', '272290', '125020', '098120', '094480', '340570', '012200', '052400',
 '333430', '053610', '452280', '215600', '117730', '066980', '012690', '018000', '007820', '003200', '082270',
 '203650', '388790', '104830', '002780', '003540', '460860', '111710', '002790', '086900', '336570', '000490',
 '000240', '168360', '085910', '003030', '242040', '299660', '274090', '159010', '035760', '321370', '004090',
 '110990', '001260', '091580', '036830', '298050', '033340', '033500', '004000', '037460', '389470', '067160',
 '001800', '025950', '056190', '013360', '064820', '365340', '067170', '214430', '046940', '105840', '013120',
 '072950', '382800', '194700', '218150', '025860', '192650', '354320', '234690', '214330', '475400', '098070',
 '230240', '015750', '052460', '053800', '025900', '138610', '452260', '030520']

ETFS = ['069500', '122630', '102110', '114800', '360750', '233740', '252670', '133690', '379810', '379800', '459580',
 '229200', '396500', '0167A0', '0193T0', '494310', '091160', '395160', '488080', '251340', '0197X0', '0195S0',
 '498400', '148020', '0193W0', '487240', '0162Z0', '381180', '232080', '472150', '462330', '0195R0', '488770',
 '395270', '0177N0', '442580', '243880', '139260', '278530', '305720', '152100', '0148J0', '228790', '226490',
 '0219E0', '390390', '0008T0', '475630', '455850', '411060', '0091P0', '426030', '0117V0', '364980', '475300',
 '466920', '458730', '105190', '0182R0', '471990', '0080G0', '469150', '0177R0', '481050', '486290', '123320',
 '0210A0', '315930', '091230', '294400', '233160', '367380', '237350', '102970', '498410', '360200', '462900',
 '445290', '292150', '102780', '457990', '462010', '449450', '465580', '277630', '161510', '091170', '497570',
 '284430', '0173Y0', '483320', '466940', '0043B0', '117700', '381170', '252710', '463250', '0183J0', '487230',
 '494300']

def unique_order(values):
    return list(dict.fromkeys(str(v).strip().upper() for v in values if str(v).strip()))

STOCKS = unique_order(STOCKS)
ETFS = unique_order(ETFS)
ALL_SYMBOLS = unique_order(STOCKS + ETFS)
ETF_SET = set(ETFS)

print(f"주식 {len(STOCKS):,}개 / ETF {len(ETFS):,}개 / 합계 {len(ALL_SYMBOLS):,}개")
print("환경변수 확인:", {key: bool(os.environ.get(key)) for key in ENV_KEYS})

if AUTO_DISCOVER_LEGACY:
    # CloudTrend 하위만 탐색하므로 다른 Drive 파일은 건드리지 않습니다.
    legacy_search_root = (
        Path("/content/drive/MyDrive/CloudTrend") if IN_COLAB else Path.cwd()
    )
    if legacy_search_root.exists():
        try:
            LEGACY_CACHE_ROOTS = list(legacy_search_root.rglob(".cloudtrend_integrated_cache_v1"))
            symbol_set = set(ALL_SYMBOLS)
            for candidate in legacy_search_root.rglob("*.csv"):
                if candidate.stem in symbol_set:
                    LEGACY_PARTIAL_FILES.setdefault(candidate.stem, candidate)
        except Exception as e:
            warnings.warn(f"기존 체크포인트 자동 탐색을 건너뜁니다: {e}")

estimated_pages = math.ceil(TARGET_COUNT / 200)
enabled_trends = sum([
    COLLECT_TOSS_INVESTOR, COLLECT_TOSS_PROGRAM, COLLECT_TOSS_SHORT,
    COLLECT_TOSS_CREDIT, COLLECT_TOSS_LENDING,
])
estimated_trend_pages = enabled_trends * math.ceil(FLOW_COUNT / 100)
print(
    f"최대 Toss 요청/종목: 캔들 {estimated_pages} + 수급 {estimated_trend_pages} "
    f"= {estimated_pages + estimated_trend_pages}회"
)
print(f"Toss workers={TOSS_WORKERS} / 완료 체크포인트={CHECKPOINT_ROOT}")


In [ ]:
# ============================================================
# 2. 공통 유틸 / Toss HTTP client / schema guard
# ============================================================

errors = []
_errors_lock = threading.Lock()

def add_error(stage, symbol=None, endpoint=None, error=None):
    with _errors_lock:
        errors.append({
            "time": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "stage": stage,
            "symbol": symbol,
            "endpoint": endpoint,
            "error": str(error)[:2000] if error is not None else None,
        })

def normalize_symbol(value):
    if value is None:
        return None
    s = str(value).strip().upper()
    return s or None

def first_text(*values):
    for v in values:
        if v is None:
            continue
        s = str(v).strip()
        if s and s.lower() not in {"none", "nan", "null"}:
            return s
    return None

def to_number(value):
    if value is None:
        return None
    if isinstance(value, bool):
        return float(value)
    s = str(value).replace(",", "").replace("₩", "").replace("%", "").strip()
    if not s or s in {"-", "--"} or s.lower() in {"none", "null", "nan"}:
        return None
    try:
        return float(s)
    except Exception:
        return None

def first_number(*values):
    for v in values:
        n = to_number(v)
        if n is not None:
            return n
    return None

def unwrap_payload(payload):
    if isinstance(payload, dict) and "result" in payload:
        return payload["result"]
    return payload

def extract_list(payload, keys=("records", "candles", "stocks", "items", "results")):
    obj = unwrap_payload(payload)
    if isinstance(obj, list):
        return obj
    if isinstance(obj, dict):
        for key in keys:
            value = obj.get(key)
            if isinstance(value, list):
                return value
    return []

def extract_cursor(payload, keys=("nextUntil", "nextBefore", "nextCursor", "cursor")):
    obj = unwrap_payload(payload)
    if isinstance(obj, dict):
        for key in keys:
            value = obj.get(key)
            if value not in (None, ""):
                return str(value)
    if isinstance(payload, dict):
        for key in keys:
            value = payload.get(key)
            if value not in (None, ""):
                return str(value)
    return None

def to_trade_date(value):
    if value is None:
        return None
    s = str(value).strip()
    if not s:
        return None
    try:
        ts = pd.Timestamp(s)
        # Toss 공식 timestamp는 +09:00 offset 포함. naive면 날짜 자체를 유지한다.
        if ts.tzinfo is not None:
            ts = ts.tz_convert("Asia/Seoul")
        return ts.strftime("%Y-%m-%d")
    except Exception:
        digits = re.sub(r"\D", "", s)
        if len(digits) >= 8:
            return f"{digits[:4]}-{digits[4:6]}-{digits[6:8]}"
        return None

def split_batches(values, size):
    for i in range(0, len(values), size):
        yield values[i:i+size]

def normalize_market(value):
    text = first_text(value)
    if text is None:
        return None
    upper = text.upper()
    if "KOSDAQ" in upper:
        return "KOSDAQ"
    if "KOSPI" in upper:
        return "KOSPI"
    if "KONEX" in upper:
        return "KONEX"
    return text

def safe_filename(text):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text))

def cache_path(stage, symbol):
    d = CACHE_ROOT / safe_filename(stage)
    d.mkdir(parents=True, exist_ok=True)
    return d / f"{safe_filename(symbol)}.pkl"

def atomic_pickle(df, path):
    """완성된 파일만 보이도록 임시 파일을 거쳐 원자적으로 교체합니다."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + f".{threading.get_ident()}.tmp")
    df.to_pickle(tmp)
    os.replace(tmp, path)

def legacy_cache_path(stage, symbol):
    for root in LEGACY_CACHE_ROOTS:
        candidate = root / safe_filename(stage) / f"{safe_filename(symbol)}.pkl"
        if candidate.exists():
            return candidate
    return None

def cached_frame(stage, symbol, loader, refresh=None):
    refresh = REFRESH_CACHE if refresh is None else refresh
    path = cache_path(stage, symbol)
    if path.exists() and not refresh:
        try:
            return pd.read_pickle(path)
        except Exception as e:
            add_error(f"cache_read_{stage}", symbol, str(path), e)
    if not refresh:
        legacy = legacy_cache_path(stage, symbol)
        if legacy is not None:
            try:
                df = pd.read_pickle(legacy)
                if isinstance(df, pd.DataFrame) and not df.empty:
                    atomic_pickle(df, path)
                    return df
            except Exception as e:
                add_error(f"legacy_cache_read_{stage}", symbol, str(legacy), e)
    try:
        df = loader()
        if df is None:
            df = pd.DataFrame()
        if not isinstance(df, pd.DataFrame):
            df = pd.DataFrame(df)
        # 빈 결과는 transient failure일 가능성이 있으므로 캐시하지 않는다.
        if not df.empty:
            atomic_pickle(df, path)
        return df
    except Exception as e:
        add_error(stage, symbol, None, e)
        if FAIL_SYMBOL_ON_STAGE_ERROR:
            raise
        return pd.DataFrame()

def validate_toss_openapi(strict=True):
    required_paths = {
        "/api/v1/candles",
        "/api/v1/stocks/{symbol}/investor-trading",
        "/api/v1/stocks/{symbol}/program-trades",
        "/api/v1/stocks/{symbol}/short-selling",
        "/api/v1/stocks/{symbol}/credit-trades",
        "/api/v1/stocks/{symbol}/securities-lending",
    }
    try:
        r = requests.get(OPENAPI_SPEC_URL, timeout=20)
        r.raise_for_status()
        spec = r.json()
        version = spec.get("info", {}).get("version")
        paths = set(spec.get("paths", {}))
        missing = sorted(required_paths - paths)
        print(f"Toss OpenAPI server schema version: {version}")
        if missing:
            msg = "필수 endpoint가 현재 OpenAPI에서 사라졌습니다: " + ", ".join(missing)
            if strict:
                raise RuntimeError(msg)
            warnings.warn(msg)
        return version
    except Exception as e:
        if strict:
            raise
        warnings.warn(f"OpenAPI schema 확인 실패: {e}")
        return None

class PacedRateLimiter:
    """여러 worker가 같은 공식 Rate Limits Group 한도를 넘지 않도록 시작 시각을 예약합니다."""

    def __init__(self, minimum_gap):
        self.minimum_gap = float(minimum_gap)
        self.next_at = 0.0
        self.lock = threading.Lock()

    def acquire(self, requested_gap=None):
        gap = max(self.minimum_gap, float(requested_gap or 0.0))
        with self.lock:
            now = time.monotonic()
            slot = max(now, self.next_at)
            self.next_at = slot + gap
        delay = slot - now
        if delay > 0:
            time.sleep(delay)

    def pause(self, seconds):
        with self.lock:
            self.next_at = max(self.next_at, time.monotonic() + max(0.0, seconds))


class TossClient:
    """Thread-safe OAuth client with per-rate-group pacing and pooled connections."""

    def __init__(self):
        self.timeout = (TOSS_CONNECT_TIMEOUT, TOSS_READ_TIMEOUT)
        self.max_retries = TOSS_MAX_RETRIES
        self._local = threading.local()
        self._token_lock = threading.Lock()
        self._token = None
        self._token_generation = 0
        self._metrics_lock = threading.Lock()
        self._request_count = 0
        self._retry_count = 0
        self._rate_limit_count = 0
        self._observed_limits = {}
        self._limiters = {
            "MARKET_DATA_CHART": PacedRateLimiter(TOSS_CHART_GAP),
            "MARKET_INDICATOR_CHART": PacedRateLimiter(TOSS_CHART_GAP),
            "STOCK_TRADING_TREND": PacedRateLimiter(TOSS_TREND_GAP),
            "STOCK": PacedRateLimiter(TOSS_STOCK_GAP),
            "MARKET_INDICATOR": PacedRateLimiter(TOSS_INDICATOR_GAP),
        }
        self.issue_token()

    def _session(self):
        session = getattr(self._local, "session", None)
        if session is None:
            session = requests.Session()
            adapter = requests.adapters.HTTPAdapter(
                pool_connections=TOSS_WORKERS + 2,
                pool_maxsize=TOSS_WORKERS + 2,
                max_retries=0,
                pool_block=True,
            )
            session.mount("https://", adapter)
            session.headers.update({"Accept": "application/json"})
            self._local.session = session
        return session

    @staticmethod
    def _group_for(endpoint):
        if "/market-indicators/" in endpoint and endpoint.endswith("/candles"):
            return "MARKET_INDICATOR_CHART"
        if endpoint.startswith("/api/v1/market-indicators/"):
            return "MARKET_INDICATOR"
        if endpoint == "/api/v1/candles":
            return "MARKET_DATA_CHART"
        if endpoint.startswith("/api/v1/stocks/") and endpoint.rsplit("/", 1)[-1] in {
            "investor-trading", "program-trades", "short-selling",
            "credit-trades", "securities-lending",
        }:
            return "STOCK_TRADING_TREND"
        return "STOCK"

    def issue_token(self, observed_generation=None):
        # client당 유효 token은 하나뿐이므로 refresh를 단일화합니다.
        with self._token_lock:
            if observed_generation is not None and observed_generation != self._token_generation:
                return
            response = requests.post(
                BASE + "/oauth2/token",
                headers={"Content-Type": "application/x-www-form-urlencoded"},
                data={
                    "grant_type": "client_credentials",
                    "client_id": CLIENT_ID,
                    "client_secret": CLIENT_SECRET,
                },
                timeout=self.timeout,
            )
            if not response.ok:
                raise RuntimeError(
                    f"Toss 토큰 발급 실패 HTTP {response.status_code}: {response.text[:1000]}"
                )
            payload = response.json()
            token = payload.get("access_token")
            if not token:
                obj = unwrap_payload(payload)
                if isinstance(obj, dict):
                    token = obj.get("access_token")
            if not token:
                raise RuntimeError("Toss token 응답에 access_token이 없습니다.")
            self._token = token
            self._token_generation += 1

    def _token_snapshot(self):
        with self._token_lock:
            return self._token, self._token_generation

    def _record_response(self, group, response):
        limit = response.headers.get("X-RateLimit-Limit")
        with self._metrics_lock:
            self._request_count += 1
            if response.status_code == 429:
                self._rate_limit_count += 1
            if limit:
                self._observed_limits[group] = limit

    def metrics(self):
        with self._metrics_lock:
            return {
                "requests": self._request_count,
                "retries": self._retry_count,
                "http_429": self._rate_limit_count,
                "observed_limits": dict(self._observed_limits),
            }

    def get(self, endpoint, minimum_gap=None, **params):
        group = self._group_for(endpoint)
        limiter = self._limiters[group]
        last_error = None

        for attempt in range(self.max_retries):
            if COLLECTION_STOP_EVENT.is_set():
                raise InterruptedError("수집 중단 요청")
            if attempt > 0:
                with self._metrics_lock:
                    self._retry_count += 1
            limiter.acquire(minimum_gap)
            token, generation = self._token_snapshot()
            try:
                response = self._session().get(
                    BASE + endpoint,
                    params=params,
                    headers={"Authorization": f"Bearer {token}"},
                    timeout=self.timeout,
                )
            except requests.RequestException as e:
                last_error = e
                time.sleep(min(6.0, 0.4 * (2 ** attempt)) + random.random() * 0.15)
                continue

            self._record_response(group, response)

            if response.status_code == 401:
                self.issue_token(observed_generation=generation)
                continue

            if response.status_code == 429:
                retry_after = response.headers.get("Retry-After")
                reset_after = response.headers.get("X-RateLimit-Reset")
                try:
                    delay = float(retry_after or reset_after or (0.75 * (attempt + 1)))
                except Exception:
                    delay = 0.75 * (attempt + 1)
                last_error = RuntimeError(f"HTTP 429, {delay:.2f}초 후 재시도")
                limiter.pause(max(0.5, delay))
                continue

            if 500 <= response.status_code < 600:
                last_error = RuntimeError(f"HTTP {response.status_code}: {response.text[:500]}")
                time.sleep(min(6.0, 0.4 * (2 ** attempt)) + random.random() * 0.15)
                continue

            if not response.ok:
                raise RuntimeError(
                    f"Toss API 오류 {endpoint} HTTP {response.status_code}: {response.text[:1200]}"
                )

            return response.json()

        raise RuntimeError(f"Toss API 반복 실패 {endpoint}: {last_error}")

# canonical server-owned spec가 바뀌었는지 실행 시점에 확인
TOSS_SCHEMA_VERSION = validate_toss_openapi(strict=False)
client = TossClient()
print("Toss OAuth 연결 성공")


In [ ]:
# ============================================================
# 3. Toss: 종목 마스터 / 일봉 / 최신 수급 endpoint 전체
# ============================================================

def get_market_master():
    output = {}
    for market in ["KOSPI", "KOSDAQ"]:
        try:
            payload = client.get(
                "/api/v1/stocks/all",
                minimum_gap=1.05,
                market=market,
                status="ACTIVE",
            )
            for record in extract_list(payload, ("stocks", "items", "records", "results")):
                if not isinstance(record, dict):
                    continue
                symbol = normalize_symbol(
                    first_text(
                        record.get("symbol"), record.get("code"),
                        record.get("ticker"), record.get("shortCode"),
                    )
                )
                if not symbol:
                    continue
                output[symbol] = {
                    **record,
                    "symbol": symbol,
                    "name": first_text(
                        record.get("name"), record.get("displayName"),
                        record.get("koreanName"), record.get("securityName"), symbol,
                    ),
                    "market": market,
                }
        except Exception as e:
            add_error("market_master", market, "/api/v1/stocks/all", e)
    return output

def get_stock_details(symbols):
    output = {}
    for batch in tqdm(list(split_batches(symbols, 200)), desc="Toss 종목 상세"):
        try:
            payload = client.get(
                "/api/v1/stocks",
                minimum_gap=0.35,
                symbols=",".join(batch),
            )
            for record in extract_list(payload, ("stocks", "items", "records", "results")):
                if not isinstance(record, dict):
                    continue
                symbol = normalize_symbol(
                    first_text(
                        record.get("symbol"), record.get("code"),
                        record.get("ticker"), record.get("shortCode"),
                    )
                )
                if symbol:
                    output[symbol] = record
        except Exception as e:
            add_error("stock_details", ",".join(batch[:10]), "/api/v1/stocks", e)
    return output

def get_candles_frame(symbol, target_count=TARGET_COUNT, is_index=False):
    endpoint = (
        f"/api/v1/market-indicators/{symbol}/candles"
        if is_index else "/api/v1/candles"
    )
    output = {}
    before = None
    seen = set()
    max_pages = max(20, (target_count // 200) + 5)

    for _ in range(max_pages):
        if COLLECTION_STOP_EVENT.is_set():
            raise InterruptedError("수집 중단 요청")
        params = {"interval": "1d", "count": min(200, target_count)}
        if not is_index:
            params.update({"symbol": symbol, "adjusted": "true"})
        if before is not None:
            params["before"] = before

        payload = client.get(endpoint, minimum_gap=TOSS_CHART_GAP, **params)
        records = extract_list(payload, ("candles", "records", "items", "results"))
        if not records:
            break

        for rec in records:
            if not isinstance(rec, dict):
                continue
            ts = first_text(rec.get("timestamp"), rec.get("date"), rec.get("tradeDate"))
            date = to_trade_date(ts)
            if not date:
                continue
            output[date] = {
                "date": date,
                "open": first_number(rec.get("openPrice")),
                "high": first_number(rec.get("highPrice")),
                "low": first_number(rec.get("lowPrice")),
                "close": first_number(rec.get("closePrice")),
                "volume": first_number(rec.get("volume")),
            }

        if len(output) >= target_count:
            break
        cursor = extract_cursor(payload, ("nextBefore",))
        if not cursor or cursor in seen:
            break
        seen.add(cursor)
        before = cursor

    if not output:
        return pd.DataFrame()
    df = pd.DataFrame(output.values()).sort_values("date").tail(target_count).reset_index(drop=True)
    return df

def fetch_trend_frame(symbol, suffix, parser, target_count=FLOW_COUNT):
    endpoint = f"/api/v1/stocks/{symbol}/{suffix}"
    output = {}
    until = None
    seen = set()
    max_pages = max(20, (target_count // 100) + 5)

    for _ in range(max_pages):
        if COLLECTION_STOP_EVENT.is_set():
            raise InterruptedError("수집 중단 요청")
        params = {"count": min(100, target_count)}
        if until is not None:
            params["until"] = until

        payload = client.get(endpoint, minimum_gap=TOSS_TREND_GAP, **params)
        records = extract_list(payload, ("records",))
        if not records:
            break

        for rec in records:
            if not isinstance(rec, dict):
                continue
            date = to_trade_date(rec.get("date"))
            if not date:
                continue
            row = parser(rec)
            row["date"] = date
            output[date] = row

        if len(output) >= target_count:
            break
        cursor = extract_cursor(payload, ("nextUntil",))
        if not cursor or cursor in seen:
            break
        seen.add(cursor)
        until = cursor

    if not output:
        return pd.DataFrame()
    return (
        pd.DataFrame(output.values())
        .sort_values("date")
        .tail(target_count)
        .reset_index(drop=True)
    )

def _volume_triplet(obj, prefix):
    obj = obj if isinstance(obj, dict) else {}
    return {
        f"{prefix}BuyVolume": first_number(obj.get("buyVolume")),
        f"{prefix}SellVolume": first_number(obj.get("sellVolume")),
        f"{prefix}NetBuyVolume": first_number(obj.get("netBuyVolume")),
    }

def parse_investor_record(record):
    individual = record.get("individual")
    foreigner = record.get("foreigner")
    institution = record.get("institution")
    other = record.get("otherCorporation")
    institution = institution if isinstance(institution, dict) else {}
    breakdown = institution.get("breakdown")
    breakdown = breakdown if isinstance(breakdown, dict) else {}
    holding = record.get("foreignerHolding")
    holding = holding if isinstance(holding, dict) else {}
    cfd = record.get("cfd")
    cfd = cfd if isinstance(cfd, dict) else {}

    out = {}
    out.update(_volume_triplet(individual, "individual"))
    out.update(_volume_triplet(foreigner, "foreign"))
    out.update(_volume_triplet(institution, "institution"))
    out.update(_volume_triplet(other, "otherCorporation"))

    detail_map = {
        "financialInvestment": "financialInvestment",
        "insurance": "insurance",
        "trust": "trust",
        "privateEquityFund": "privateEquityFund",
        "bank": "bank",
        "otherFinancialInstitution": "otherFinancialInstitution",
        "pensionFund": "pensionFund",
    }
    for api_key, prefix in detail_map.items():
        part = breakdown.get(api_key)
        part = part if isinstance(part, dict) else {}
        out[f"{prefix}NetBuyVolume"] = first_number(part.get("netBuyVolume"))

    rate = first_number(holding.get("holdingRate"))
    out.update({
        "foreignHoldingQuantity": first_number(holding.get("holdingQuantity")),
        "foreignHoldingLimitQuantity": first_number(holding.get("limitQuantity")),
        "foreignHoldingRate": rate,
        "foreignHoldingRatePct": rate * 100 if rate is not None else None,
        "cfdBuyBalanceQuantity": first_number(cfd.get("buyBalanceQuantity")),
        "cfdBuyBalanceRate": first_number(cfd.get("buyBalanceRate")),
        "cfdSellBalanceQuantity": first_number(cfd.get("sellBalanceQuantity")),
        "cfdSellBalanceRate": first_number(cfd.get("sellBalanceRate")),
        "investorUpdatedAt": first_text(record.get("updatedAt")),
    })
    return out

def parse_program_record(record):
    arb = record.get("arbitrage")
    non = record.get("nonArbitrage")
    arb = arb if isinstance(arb, dict) else {}
    non = non if isinstance(non, dict) else {}
    a = first_number(arb.get("netBuyVolume"))
    n = first_number(non.get("netBuyVolume"))
    total = None if a is None and n is None else (a or 0) + (n or 0)
    return {
        "programArbitrageBuyVolume": first_number(arb.get("buyVolume")),
        "programArbitrageSellVolume": first_number(arb.get("sellVolume")),
        "programArbitrageNetBuyVolume": a,
        "programNonArbitrageBuyVolume": first_number(non.get("buyVolume")),
        "programNonArbitrageSellVolume": first_number(non.get("sellVolume")),
        "programNonArbitrageNetBuyVolume": n,
        "programNetBuyVolume": total,
    }

def parse_short_record(record):
    return {
        "shortSellingVolume": first_number(record.get("shortSellingVolume")),
        "shortSellingAmount": first_number(record.get("shortSellingAmount")),
        "shortSellingVolumeRate": first_number(record.get("shortSellingVolumeRate")),
        "shortSellingAmountRate": first_number(record.get("shortSellingAmountRate")),
        "shortUpdatedAt": first_text(record.get("updatedAt")),
    }

def _credit_detail(record, api_key, prefix):
    obj = record.get(api_key)
    obj = obj if isinstance(obj, dict) else {}
    return {
        f"{prefix}NewQuantity": first_number(obj.get("newQuantity")),
        f"{prefix}ReturnQuantity": first_number(obj.get("returnQuantity")),
        f"{prefix}BalanceQuantity": first_number(obj.get("balanceQuantity")),
        f"{prefix}BalanceRate": first_number(obj.get("balanceRate")),
        f"{prefix}TradingRate": first_number(obj.get("tradingRate")),
    }

def parse_credit_record(record):
    out = {}
    out.update(_credit_detail(record, "marginLoan", "marginLoan"))
    out.update(_credit_detail(record, "stockLoan", "stockLoan"))
    out["creditUpdatedAt"] = first_text(record.get("updatedAt"))
    return out

def parse_lending_record(record):
    return {
        "lendingExecutionQuantity": first_number(record.get("executionQuantity")),
        "lendingRepaymentQuantity": first_number(record.get("repaymentQuantity")),
        "lendingBalanceQuantity": first_number(record.get("balanceQuantity")),
        "lendingBalanceAmount": first_number(record.get("balanceAmount")),
        "lendingUpdatedAt": first_text(record.get("updatedAt")),
    }

def get_market_investor_flow_frame(symbol, target_count=FLOW_COUNT):
    endpoint = f"/api/v1/market-indicators/{symbol}/investor-trading"
    output = {}
    until = None
    seen = set()
    max_pages = max(20, (target_count // 100) + 5)

    for _ in range(max_pages):
        if COLLECTION_STOP_EVENT.is_set():
            raise InterruptedError("수집 중단 요청")
        params = {"interval": "1d", "count": min(100, target_count)}
        if until:
            params["until"] = until
        payload = client.get(endpoint, minimum_gap=TOSS_TREND_GAP, **params)
        records = extract_list(payload, ("records",))
        if not records:
            break

        for rec in records:
            if not isinstance(rec, dict):
                continue
            date = to_trade_date(rec.get("date"))
            if not date:
                continue

            def net_amount(key):
                obj = rec.get(key)
                obj = obj if isinstance(obj, dict) else {}
                buy = first_number(obj.get("buyAmount"))
                sell = first_number(obj.get("sellAmount"))
                return buy - sell if buy is not None and sell is not None else None

            output[date] = {
                "date": date,
                "foreignNetBuyValue": net_amount("foreigner"),
                "institutionNetBuyValue": net_amount("institution"),
                "individualNetBuyValue": net_amount("individual"),
                "otherCorporationNetBuyValue": net_amount("otherCorporation"),
                "marketFlowUpdatedAt": first_text(rec.get("updatedAt")),
            }

        if len(output) >= target_count:
            break
        cursor = extract_cursor(payload, ("nextUntil",))
        if not cursor or cursor in seen:
            break
        seen.add(cursor)
        until = cursor

    if not output:
        return pd.DataFrame()
    return pd.DataFrame(output.values()).sort_values("date").tail(target_count).reset_index(drop=True)

# Toss 마스터
market_master = get_market_master()
stock_details = get_stock_details(ALL_SYMBOLS)

symbol_info = {}
for symbol in ALL_SYMBOLS:
    master = market_master.get(symbol, {})
    detail = stock_details.get(symbol, {})
    symbol_info[symbol] = {
        "symbol": symbol,
        "name": first_text(detail.get("name"), master.get("name"), symbol),
        "market": first_text(
            normalize_market(detail.get("market")),
            normalize_market(master.get("market")),
        ),
        "securityType": "ETF" if symbol in ETF_SET else "STOCK",
        # 참고용 현재 발행주식수. 역사적 marketCap 계산에는 사용 금지.
        "sharesOutstandingCurrent": first_number(
            detail.get("sharesOutstanding"), master.get("sharesOutstanding")
        ),
    }

print(f"Toss 시장 마스터 {len(market_master):,}개 / 상세 {len(stock_details):,}개")


In [ ]:
# ============================================================
# 4. KRX exact data: 거래대금/시총/순매수금액/ETF 기준정보
# ============================================================

_krx_last_call = 0.0
_krx_request_lock = threading.Lock()

def krx_wait():
    global _krx_last_call
    elapsed = time.monotonic() - _krx_last_call
    if elapsed < KRX_MIN_GAP:
        time.sleep(KRX_MIN_GAP - elapsed)
    _krx_last_call = time.monotonic()

def krx_retry(func, *args, attempts=4, **kwargs):
    last = None
    for i in range(attempts):
        try:
            # pykrx 세션은 worker 간 공유에 안전하다고 보장되지 않으므로 실제 HTTP
            # 호출 전체를 한 번에 하나만 실행합니다. Toss 요청은 다른 thread에서 계속됩니다.
            with _krx_request_lock:
                if COLLECTION_STOP_EVENT.is_set():
                    raise InterruptedError("수집 중단 요청")
                krx_wait()
                return func(*args, **kwargs)
        except Exception as e:
            last = e
            if isinstance(e, InterruptedError):
                raise
            time.sleep(min(8.0, 0.8 * (2 ** i)) + random.random() * 0.3)
    raise RuntimeError(f"KRX/pykrx 반복 실패 {getattr(func, '__name__', func)}: {last}")

def frame_with_date_index(df):
    if df is None or len(df) == 0:
        return pd.DataFrame()
    out = df.copy()
    if "date" not in out.columns:
        idx = pd.to_datetime(out.index, errors="coerce")
        out.insert(0, "date", idx.strftime("%Y-%m-%d"))
    else:
        out["date"] = pd.to_datetime(out["date"], errors="coerce").dt.strftime("%Y-%m-%d")
    return out.dropna(subset=["date"]).reset_index(drop=True)

def numeric_series(series):
    return pd.to_numeric(
        series.astype(str).str.replace(",", "", regex=False).replace({"-": None, "": None, "nan": None}),
        errors="coerce",
    )

def get_krx_market_frame(symbol, start_yyyymmdd, end_yyyymmdd):
    df = krx_retry(
        stock.get_market_cap_by_date,
        start_yyyymmdd,
        end_yyyymmdd,
        symbol,
    )
    df = frame_with_date_index(df)
    if df.empty:
        return df
    rename = {
        "시가총액": "krxMarketCap",
        "거래량": "krxVolume",
        "거래대금": "krxTradingValue",
        "상장주식수": "krxListedShares",
    }
    df = df.rename(columns=rename)
    keep = ["date"] + [c for c in rename.values() if c in df.columns]
    df = df[keep].copy()
    for c in keep:
        if c != "date":
            df[c] = numeric_series(df[c])
    return df

KRX_INVESTOR_DETAIL_MAP = {
    "금융투자": "financialInvestmentNetBuyValue",
    "보험": "insuranceNetBuyValue",
    "투신": "trustNetBuyValue",
    "사모": "privateEquityFundNetBuyValue",
    "은행": "bankNetBuyValue",
    "기타금융": "otherFinancialInstitutionNetBuyValue",
    "연기금": "pensionFundNetBuyValue",
    "연기금 등": "pensionFundNetBuyValue",
    "기타법인": "otherCorporationNetBuyValue",
    "개인": "individualNetBuyValue",
    "외국인": "registeredForeignNetBuyValue",
    "외국인합계": "foreignNetBuyValue",
    "기타외국인": "otherForeignNetBuyValue",
}

def get_krx_stock_investor_value_frame(symbol, start_yyyymmdd, end_yyyymmdd):
    # detail=True -> 기관 7개 세부분류를 KRW 순매수 금액으로 확보.
    try:
        df = krx_retry(
            stock.get_market_trading_value_by_date,
            start_yyyymmdd,
            end_yyyymmdd,
            symbol,
            on="순매수",
            detail=True,
        )
    except Exception:
        df = krx_retry(
            stock.get_market_trading_value_by_date,
            start_yyyymmdd,
            end_yyyymmdd,
            symbol,
            on="순매수",
            detail=False,
        )

    df = frame_with_date_index(df)
    if df.empty:
        return df

    out = pd.DataFrame({"date": df["date"]})
    for kr_name, out_name in KRX_INVESTOR_DETAIL_MAP.items():
        if kr_name in df.columns and out_name not in out.columns:
            out[out_name] = numeric_series(df[kr_name])

    # detail=True 응답은 기관합계 열이 없으므로 7개 기관 세부 금액의 합으로 정확히 재구성.
    detail_cols = [
        "financialInvestmentNetBuyValue",
        "insuranceNetBuyValue",
        "trustNetBuyValue",
        "privateEquityFundNetBuyValue",
        "bankNetBuyValue",
        "otherFinancialInstitutionNetBuyValue",
        "pensionFundNetBuyValue",
    ]
    available = [c for c in detail_cols if c in out.columns]
    if available:
        out["institutionNetBuyValue"] = out[available].sum(axis=1, min_count=1)
    elif "기관합계" in df.columns:
        out["institutionNetBuyValue"] = numeric_series(df["기관합계"])

    # detail=True이면 외국인과 기타외국인이 분리되므로 KRX 외국인합계를 재구성.
    if "foreignNetBuyValue" not in out.columns:
        foreign_parts = [
            c for c in ["registeredForeignNetBuyValue", "otherForeignNetBuyValue"]
            if c in out.columns
        ]
        if foreign_parts:
            out["foreignNetBuyValue"] = out[foreign_parts].sum(axis=1, min_count=1)
        else:
            out["foreignNetBuyValue"] = pd.NA

    if "institutionNetBuyValue" not in out.columns:
        out["institutionNetBuyValue"] = pd.NA

    return out

def get_krx_etf_reference_frame(symbol, start_yyyymmdd, end_yyyymmdd):
    """
    우선 pykrx KRX raw ETF 일별 API(MDCSTAT04501)를 사용:
    NAV / 거래대금 / 시가총액 / 순자산총액 / 상장좌수 / 기초지수.
    실패하면 public get_etf_ohlcv_by_date로 NAV/거래대금/기초지수까지만 fallback.
    """
    base = pd.DataFrame()

    try:
        isin = krx_retry(get_etx_isin, symbol)
        raw = krx_retry(개별종목시세_ETF().fetch, start_yyyymmdd, end_yyyymmdd, isin)
        if raw is not None and not raw.empty:
            out = pd.DataFrame()
            out["date"] = pd.to_datetime(raw["TRD_DD"], errors="coerce").dt.strftime("%Y-%m-%d")
            raw_map = {
                "LST_NAV": "etfNav",
                "ACC_TRDVAL": "etfTradingValue",
                "MKTCAP": "etfMarketCap",
                "INVSTASST_NETASST_TOTAMT": "etfNetAssetTotalAmount",
                "LIST_SHRS": "etfListedUnits",
                "OBJ_STKPRC_IDX": "etfUnderlyingIndexClose",
            }
            for src, dst in raw_map.items():
                if src in raw.columns:
                    out[dst] = numeric_series(raw[src])
            if "IDX_IND_NM" in raw.columns:
                out["etfUnderlyingIndexName"] = raw["IDX_IND_NM"].astype("string").str.strip()
            base = out.dropna(subset=["date"]).sort_values("date").drop_duplicates("date", keep="last")
    except Exception as e:
        add_error("krx_etf_raw", symbol, "MDCSTAT04501", e)

    if base.empty:
        try:
            pub = krx_retry(stock.get_etf_ohlcv_by_date, start_yyyymmdd, end_yyyymmdd, symbol)
            pub = frame_with_date_index(pub)
            if not pub.empty:
                base = pd.DataFrame({"date": pub["date"]})
                if "NAV" in pub.columns:
                    base["etfNav"] = numeric_series(pub["NAV"])
                if "거래대금" in pub.columns:
                    base["etfTradingValue"] = numeric_series(pub["거래대금"])
                if "기초지수" in pub.columns:
                    base["etfUnderlyingIndexClose"] = numeric_series(pub["기초지수"])
        except Exception as e:
            add_error("krx_etf_public", symbol, "get_etf_ohlcv_by_date", e)

    # 괴리율
    try:
        dev = krx_retry(stock.get_etf_price_deviation, start_yyyymmdd, end_yyyymmdd, symbol)
        dev = frame_with_date_index(dev)
        if not dev.empty:
            d = pd.DataFrame({"date": dev["date"]})
            if "괴리율" in dev.columns:
                d["etfPremiumDiscountRate"] = numeric_series(dev["괴리율"])
            if base.empty:
                base = d
            else:
                base = base.merge(d, on="date", how="outer")
    except Exception as e:
        add_error("krx_etf_deviation", symbol, "get_etf_price_deviation", e)

    # 추적오차율
    try:
        te = krx_retry(stock.get_etf_tracking_error, start_yyyymmdd, end_yyyymmdd, symbol)
        te = frame_with_date_index(te)
        if not te.empty:
            t = pd.DataFrame({"date": te["date"]})
            candidate = next((c for c in ["추적오차율", "추적오차"] if c in te.columns), None)
            if candidate:
                t["etfTrackingErrorRate"] = numeric_series(te[candidate])
            if base.empty:
                base = t
            else:
                base = base.merge(t, on="date", how="outer")
    except Exception as e:
        add_error("krx_etf_tracking_error", symbol, "get_etf_tracking_error", e)

    if base.empty:
        return base
    return base.sort_values("date").drop_duplicates("date", keep="last").reset_index(drop=True)

def get_krx_etf_investor_value_frame(symbol, start_yyyymmdd, end_yyyymmdd):
    # pykrx 함수명은 upstream 철자 그대로 indivisual.
    df = krx_retry(
        etx_wrap.get_indivisual_trading_volume_and_value_by_date,
        start_yyyymmdd,
        end_yyyymmdd,
        symbol,
        "거래대금",
        "순매수",
    )
    df = frame_with_date_index(df)
    if df.empty:
        return df

    out = pd.DataFrame({"date": df["date"]})
    mapping = {
        "기관": "institutionNetBuyValue",
        "기관합계": "institutionNetBuyValue",
        "기타법인": "otherCorporationNetBuyValue",
        "개인": "individualNetBuyValue",
        "외국인": "foreignNetBuyValue",
        "외국인합계": "foreignNetBuyValue",
    }
    for src, dst in mapping.items():
        if src in df.columns and dst not in out.columns:
            out[dst] = numeric_series(df[src])
    return out


In [ ]:
# ============================================================
# 5. 종목별 Toss + KRX 통합
# ============================================================

def merge_on_date(base, frames):
    out = base.copy()
    for frame in frames:
        if frame is None or frame.empty:
            continue
        f = frame.copy()
        if "date" not in f.columns:
            continue
        # 같은 이름이 의도치 않게 겹치면 기존 base가 아닌 새 source를 명시적으로 설계해야 한다.
        overlap = (set(out.columns) & set(f.columns)) - {"date"}
        if overlap:
            raise RuntimeError(f"merge column 충돌: {sorted(overlap)}")
        out = out.merge(f, on="date", how="left")
    return out

def cached_toss_trend(symbol, stage, suffix, parser):
    return cached_frame(
        stage,
        symbol,
        lambda: fetch_trend_frame(symbol, suffix, parser, FLOW_COUNT),
    )

def build_symbol_frame(symbol):
    info = symbol_info.get(symbol, {})
    is_etf = symbol in ETF_SET

    candles = cached_frame(
        "toss_candles",
        symbol,
        lambda: get_candles_frame(symbol, TARGET_COUNT, False),
    )
    if candles.empty:
        raise RuntimeError("Toss candle 데이터 없음")

    start = candles["date"].min().replace("-", "")
    end = candles["date"].max().replace("-", "")

    frames = []

    # Priority 3: Toss 투자자별 매매 + 보유량
    if COLLECT_TOSS_INVESTOR:
        frames.append(cached_toss_trend(
            symbol, "toss_investor", "investor-trading", parse_investor_record
        ))

    # Priority 5: 프로그램매매
    if COLLECT_TOSS_PROGRAM:
        frames.append(cached_toss_trend(
            symbol, "toss_program", "program-trades", parse_program_record
        ))

    # Priority 4: 공매도 / 신용 / 대차
    if COLLECT_TOSS_SHORT:
        frames.append(cached_toss_trend(
            symbol, "toss_short", "short-selling", parse_short_record
        ))
    if COLLECT_TOSS_CREDIT:
        frames.append(cached_toss_trend(
            symbol, "toss_credit", "credit-trades", parse_credit_record
        ))
    if COLLECT_TOSS_LENDING:
        frames.append(cached_toss_trend(
            symbol, "toss_lending", "securities-lending", parse_lending_record
        ))

    # Toss 네트워크 대기를 먼저 겹쳐 처리한 후 KRX 구간으로 진입합니다.
    # KRX 자체는 krx_retry 내부 lock에 의해 전체 worker에서 1개씩만 호출됩니다.
    if COLLECT_KRX_MARKET and not is_etf:
        frames.append(cached_frame(
            "krx_market",
            symbol,
            lambda: get_krx_market_frame(symbol, start, end),
        ))

    if COLLECT_KRX_INVESTOR_VALUE and not is_etf:
        frames.append(cached_frame(
            "krx_investor_value",
            symbol,
            lambda: get_krx_stock_investor_value_frame(symbol, start, end),
        ))

    if is_etf and COLLECT_KRX_ETF:
        frames.append(cached_frame(
            "krx_etf_reference",
            symbol,
            lambda: get_krx_etf_reference_frame(symbol, start, end),
        ))
        frames.append(cached_frame(
            "krx_etf_investor_value",
            symbol,
            lambda: get_krx_etf_investor_value_frame(symbol, start, end),
        ))

    df = merge_on_date(candles, frames)

    # 메타데이터
    df.insert(0, "symbol", symbol)
    df.insert(1, "name", first_text(info.get("name"), symbol))
    df.insert(2, "market", first_text(info.get("market")))
    df.insert(3, "securityType", "ETF" if is_etf else "STOCK")
    df["priceSource"] = "TOSS_ADJUSTED_CANDLE"

    # Priority 1 canonical exact fields.
    if is_etf:
        exact_tv = df["etfTradingValue"] if "etfTradingValue" in df.columns else pd.Series(pd.NA, index=df.index)
        exact_cap = df["etfMarketCap"] if "etfMarketCap" in df.columns else pd.Series(pd.NA, index=df.index)
        listed = df["etfListedUnits"] if "etfListedUnits" in df.columns else pd.Series(pd.NA, index=df.index)
        exact_source = "KRX_ETF"
    else:
        exact_tv = df["krxTradingValue"] if "krxTradingValue" in df.columns else pd.Series(pd.NA, index=df.index)
        exact_cap = df["krxMarketCap"] if "krxMarketCap" in df.columns else pd.Series(pd.NA, index=df.index)
        listed = df["krxListedShares"] if "krxListedShares" in df.columns else pd.Series(pd.NA, index=df.index)
        exact_source = "KRX_STOCK"

    exact_tv_num = pd.to_numeric(exact_tv, errors="coerce")
    close_num = pd.to_numeric(df["close"], errors="coerce")
    volume_num = pd.to_numeric(df["volume"], errors="coerce")

    df["tradingValue"] = exact_tv_num
    proxy_mask = df["tradingValue"].isna()
    df.loc[proxy_mask, "tradingValue"] = close_num[proxy_mask] * volume_num[proxy_mask]
    df["tradingValueSource"] = exact_source
    df.loc[proxy_mask, "tradingValueSource"] = "CLOSE_X_VOLUME_PROXY"

    # 과거 시가총액은 KRX exact가 없으면 null. 현재 발행주식수×과거종가 금지.
    df["marketCap"] = pd.to_numeric(exact_cap, errors="coerce")
    df["listedShares"] = pd.to_numeric(listed, errors="coerce")
    df["marketCapSource"] = pd.NA
    df.loc[df["marketCap"].notna(), "marketCapSource"] = exact_source

    # KRX investor frame이 이미 canonical 이름으로 들어옴.
    for c in ["foreignNetBuyValue", "institutionNetBuyValue", "individualNetBuyValue", "otherCorporationNetBuyValue"]:
        if c not in df.columns:
            df[c] = pd.NA

    investor_value_present = df[["foreignNetBuyValue", "institutionNetBuyValue"]].notna().any(axis=1)
    df["investorValueSource"] = pd.NA
    df.loc[investor_value_present, "investorValueSource"] = "KRX_ETF" if is_etf else "KRX_STOCK"

    # Toss 종목 수급은 KRX+NXT 통합 거래량. 금액으로 환산하지 않는다.
    investor_volume_cols = [
        "individualNetBuyVolume", "foreignNetBuyVolume",
        "institutionNetBuyVolume", "otherCorporationNetBuyVolume",
    ]
    present_cols = [c for c in investor_volume_cols if c in df.columns]
    df["investorVolumeSource"] = pd.NA
    if present_cols:
        m = df[present_cols].notna().any(axis=1)
        df.loc[m, "investorVolumeSource"] = "TOSS_KRX_NXT"

    df["programTradeSource"] = pd.NA
    if "programNetBuyVolume" in df.columns:
        df.loc[df["programNetBuyVolume"].notna(), "programTradeSource"] = "TOSS_KRX_ONLY"

    return df

def _valid_completed_frame(frame, expected_symbol=None):
    required = {"symbol", "date", "open", "high", "low", "close", "volume"}
    if not isinstance(frame, pd.DataFrame) or frame.empty or not required.issubset(frame.columns):
        return False
    symbols = set(frame["symbol"].astype(str).str.upper().unique())
    return expected_symbol in symbols if expected_symbol else bool(symbols)


def load_completed_frames():
    """v2 shard와 기존 종목별 CSV를 읽어 완료 symbol별 최신 프레임을 반환합니다."""
    completed = {}

    for path in sorted(CHECKPOINT_ROOT.glob("checkpoint_*.pkl")):
        try:
            shard = pd.read_pickle(path)
            if not _valid_completed_frame(shard):
                raise ValueError("필수 컬럼이 없는 checkpoint")
            for symbol, part in shard.groupby("symbol", sort=False):
                symbol = normalize_symbol(symbol)
                if symbol in ALL_SYMBOLS:
                    completed[symbol] = part.reset_index(drop=True)
        except Exception as e:
            add_error("checkpoint_read", None, str(path), e)

    # 이전 Work 코드가 남긴 PARTIAL_ROOT/{symbol}.csv도 자동 승계합니다.
    for symbol, path in LEGACY_PARTIAL_FILES.items():
        if symbol in completed:
            continue
        try:
            part = pd.read_csv(path, dtype={"symbol": "string"}, low_memory=False)
            if _valid_completed_frame(part, symbol):
                completed[symbol] = part.reset_index(drop=True)
        except Exception as e:
            add_error("legacy_partial_read", symbol, str(path), e)
    return completed


def flush_checkpoint(buffer):
    if not buffer:
        return None
    shard = pd.concat(buffer, ignore_index=True, sort=False)
    symbols = sorted(shard["symbol"].astype(str).unique())
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
    path = CHECKPOINT_ROOT / (
        f"checkpoint_{stamp}_{time.time_ns()}_{symbols[0]}_{symbols[-1]}.pkl"
    )
    atomic_pickle(shard, path)
    buffer.clear()
    return path


def timed_build(symbol):
    started = time.monotonic()
    frame = build_symbol_frame(symbol)
    return symbol, frame, time.monotonic() - started


COLLECTION_STOP_EVENT.clear()
completed_map = load_completed_frames()
symbol_frames = list(completed_map.values())
completed_symbols = set(completed_map)
pending_symbols = [s for s in ALL_SYMBOLS if s not in completed_symbols]
checkpoint_buffer = []

print(
    f"재개 상태: 완료 {len(completed_symbols):,} / 남은 {len(pending_symbols):,} / "
    f"workers {TOSS_WORKERS}"
)

executor = ThreadPoolExecutor(max_workers=TOSS_WORKERS, thread_name_prefix="cloudtrend")
futures = {executor.submit(timed_build, symbol): symbol for symbol in pending_symbols}
progress = tqdm(
    total=len(ALL_SYMBOLS),
    initial=len(completed_symbols),
    desc="CloudTrend Toss+KRX 고속 통합 수집",
)

try:
    for future in as_completed(futures):
        symbol = futures[future]
        try:
            _, part, elapsed = future.result()
            if not part.empty:
                symbol_frames.append(part)
                checkpoint_buffer.append(part)
                if len(checkpoint_buffer) >= CHECKPOINT_EVERY:
                    flush_checkpoint(checkpoint_buffer)
            progress.set_postfix_str(
                f"last={symbol} {elapsed:.1f}s, req={client.metrics()['requests']:,}, "
                f"429={client.metrics()['http_429']}"
            )
        except InterruptedError:
            pass
        except Exception as e:
            add_error("build_symbol", symbol, None, e)
        finally:
            progress.update(1)

    flush_checkpoint(checkpoint_buffer)
    executor.shutdown(wait=True)
except KeyboardInterrupt:
    COLLECTION_STOP_EVENT.set()
    for future in futures:
        future.cancel()
    flush_checkpoint(checkpoint_buffer)
    executor.shutdown(wait=False, cancel_futures=True)
    progress.close()
    print("중단됨. 완료된 종목은 Drive 체크포인트에 저장되었습니다.")
    print("같은 노트북을 다시 실행하면 완료 종목을 자동으로 건너뜁니다.")
    raise
finally:
    progress.close()

if not symbol_frames:
    raise RuntimeError("개별 종목/ETF 데이터가 생성되지 않았습니다.")

successful_symbols = {
    normalize_symbol(frame["symbol"].iloc[0])
    for frame in symbol_frames
    if _valid_completed_frame(frame)
}
missing_symbols = [s for s in ALL_SYMBOLS if s not in successful_symbols]
if missing_symbols:
    raise RuntimeError(
        f"{len(missing_symbols):,}개 symbol 수집이 완료되지 않았습니다. "
        "성공분은 체크포인트에 저장했으므로 이 셀을 다시 실행하세요. "
        f"앞 20개: {missing_symbols[:20]}"
    )

non_index_df = pd.concat(symbol_frames, ignore_index=True, sort=False)
print(
    f"개별 종목/ETF: {len(non_index_df):,}행 / "
    f"{non_index_df['symbol'].nunique():,} symbols"
)
print("Toss HTTP 통계:", client.metrics())


In [ ]:
# ============================================================
# 6. KOSPI/KOSDAQ 지수 — Toss 일봉 + 실제 시장 투자자 매매대금
# ============================================================

def build_index_frame(symbol, name):
    candles = cached_frame(
        "toss_index_candles",
        symbol,
        lambda: get_candles_frame(symbol, TARGET_COUNT, True),
    )
    if candles.empty:
        return pd.DataFrame()

    flow = cached_frame(
        "toss_index_flow",
        symbol,
        lambda: get_market_investor_flow_frame(symbol, FLOW_COUNT),
    )
    df = merge_on_date(candles, [flow])
    df.insert(0, "symbol", symbol)
    df.insert(1, "name", name)
    df.insert(2, "market", "INDEX")
    df.insert(3, "securityType", "INDEX")
    df["priceSource"] = "TOSS_MARKET_INDICATOR"
    df["tradingValue"] = 0.0
    df["tradingValueSource"] = "NOT_APPLICABLE_INDEX"
    df["marketCap"] = pd.NA
    df["listedShares"] = pd.NA
    df["marketCapSource"] = pd.NA
    df["investorValueSource"] = "TOSS_MARKET_INDICATOR_KRW"
    df["investorVolumeSource"] = pd.NA
    df["programTradeSource"] = pd.NA
    return df

index_frames = []
for symbol, name in [("KOSPI", "코스피"), ("KOSDAQ", "코스닥")]:
    try:
        f = build_index_frame(symbol, name)
        if not f.empty:
            index_frames.append(f)
    except Exception as e:
        add_error("index_build", symbol, None, e)

df = pd.concat(
    [non_index_df] + index_frames,
    ignore_index=True,
    sort=False,
)

print(f"지수 포함 전체: {len(df):,}행 / {df['symbol'].nunique():,} symbols")


In [ ]:
# ============================================================
# 7. 컬럼 계약 / 무결성 검증 / 저장
# ============================================================

CANONICAL_COLUMNS = [
    "symbol", "name", "market", "securityType", "date",
    "open", "high", "low", "close", "volume",
    "tradingValue", "marketCap",
    "foreignNetBuyValue", "institutionNetBuyValue",
]

EXTENDED_COLUMNS = [
    # KRX historical exact / audit
    "listedShares", "krxVolume", "krxTradingValue", "krxMarketCap", "krxListedShares",
    "individualNetBuyValue", "otherCorporationNetBuyValue",
    "registeredForeignNetBuyValue", "otherForeignNetBuyValue",
    "financialInvestmentNetBuyValue", "insuranceNetBuyValue", "trustNetBuyValue",
    "privateEquityFundNetBuyValue", "bankNetBuyValue",
    "otherFinancialInstitutionNetBuyValue", "pensionFundNetBuyValue",

    # Toss top-level investor volume
    "individualBuyVolume", "individualSellVolume", "individualNetBuyVolume",
    "foreignBuyVolume", "foreignSellVolume", "foreignNetBuyVolume",
    "institutionBuyVolume", "institutionSellVolume", "institutionNetBuyVolume",
    "otherCorporationBuyVolume", "otherCorporationSellVolume", "otherCorporationNetBuyVolume",

    # Toss institution breakdown
    "financialInvestmentNetBuyVolume", "insuranceNetBuyVolume", "trustNetBuyVolume",
    "privateEquityFundNetBuyVolume", "bankNetBuyVolume",
    "otherFinancialInstitutionNetBuyVolume", "pensionFundNetBuyVolume",

    # Toss foreign holding / CFD
    "foreignHoldingQuantity", "foreignHoldingLimitQuantity",
    "foreignHoldingRate", "foreignHoldingRatePct",
    "cfdBuyBalanceQuantity", "cfdBuyBalanceRate",
    "cfdSellBalanceQuantity", "cfdSellBalanceRate",
    "investorUpdatedAt",

    # Toss program
    "programArbitrageBuyVolume", "programArbitrageSellVolume", "programArbitrageNetBuyVolume",
    "programNonArbitrageBuyVolume", "programNonArbitrageSellVolume", "programNonArbitrageNetBuyVolume",
    "programNetBuyVolume",

    # Toss short-selling
    "shortSellingVolume", "shortSellingAmount",
    "shortSellingVolumeRate", "shortSellingAmountRate", "shortUpdatedAt",

    # Toss credit
    "marginLoanNewQuantity", "marginLoanReturnQuantity", "marginLoanBalanceQuantity",
    "marginLoanBalanceRate", "marginLoanTradingRate",
    "stockLoanNewQuantity", "stockLoanReturnQuantity", "stockLoanBalanceQuantity",
    "stockLoanBalanceRate", "stockLoanTradingRate", "creditUpdatedAt",

    # Toss securities lending
    "lendingExecutionQuantity", "lendingRepaymentQuantity",
    "lendingBalanceQuantity", "lendingBalanceAmount", "lendingUpdatedAt",

    # KRX ETF
    "etfNav", "etfTradingValue", "etfMarketCap",
    "etfNetAssetTotalAmount", "etfListedUnits",
    "etfUnderlyingIndexName", "etfUnderlyingIndexClose",
    "etfPremiumDiscountRate", "etfTrackingErrorRate",

    # provenance
    "priceSource", "tradingValueSource", "marketCapSource",
    "investorValueSource", "investorVolumeSource", "programTradeSource",
    "marketFlowUpdatedAt",
]

FINAL_COLUMNS = CANONICAL_COLUMNS + EXTENDED_COLUMNS

for c in FINAL_COLUMNS:
    if c not in df.columns:
        df[c] = pd.NA

df = df[FINAL_COLUMNS].copy()
df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.strftime("%Y-%m-%d")

# 숫자열은 문자열 timestamp/source/name 등을 제외하고 명시적으로 정리
TEXT_COLUMNS = {
    "symbol", "name", "market", "securityType", "date",
    "investorUpdatedAt", "shortUpdatedAt", "creditUpdatedAt", "lendingUpdatedAt",
    "etfUnderlyingIndexName",
    "priceSource", "tradingValueSource", "marketCapSource",
    "investorValueSource", "investorVolumeSource", "programTradeSource",
    "marketFlowUpdatedAt",
}
for c in FINAL_COLUMNS:
    if c not in TEXT_COLUMNS:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# 필수 가격 데이터가 없는 행 제거
required = ["symbol", "date", "open", "high", "low", "close", "volume"]
bad = df[required].isna().any(axis=1)
if bad.any():
    add_error("validation", None, None, f"필수값 누락 {int(bad.sum()):,}행 제거")
    df = df.loc[~bad].copy()

# symbol-date 중복 제거
dup = df.duplicated(["symbol", "date"], keep="last")
if dup.any():
    add_error("validation", None, None, f"중복 {int(dup.sum()):,}행 제거")
    df = df.loc[~dup].copy()

df = df.sort_values(["symbol", "date"]).reset_index(drop=True)
non_index_mask = df["securityType"] != "INDEX"

# 금지된 추정치 탐지:
# 1) marketCap은 source가 KRX인 행만 허용
bad_cap = (
    non_index_mask
    & df["marketCap"].notna()
    & ~df["marketCapSource"].isin(["KRX_STOCK", "KRX_ETF"])
)
if bad_cap.any():
    raise RuntimeError("무결성 위반: KRX exact가 아닌 marketCap이 존재합니다.")

# 2) 종목 foreign/institution 금액은 KRX source인 행만 허용
value_present = (
    non_index_mask
    & df[["foreignNetBuyValue", "institutionNetBuyValue"]].notna().any(axis=1)
)
bad_value = value_present & ~df["investorValueSource"].isin(["KRX_STOCK", "KRX_ETF"])
if bad_value.any():
    raise RuntimeError("무결성 위반: KRX exact가 아닌 종목별 순매수금액이 존재합니다.")

# 3) 거래대금은 exact KRX 또는 명시적 close*volume proxy만 허용
allowed_tv = {"KRX_STOCK", "KRX_ETF", "CLOSE_X_VOLUME_PROXY", "NOT_APPLICABLE_INDEX"}
if (~df["tradingValueSource"].isin(allowed_tv)).any():
    raise RuntimeError("무결성 위반: 알 수 없는 tradingValueSource가 존재합니다.")

def coverage(series):
    return float(series.notna().mean() * 100) if len(series) else 0.0

summary_rows = [
    ["전체 행", len(df)],
    ["전체 symbol", df["symbol"].nunique()],
    ["주식/ETF symbol", df.loc[non_index_mask, "symbol"].nunique()],
    ["KRX exact 거래대금 coverage %", coverage(df.loc[non_index_mask, "tradingValue"].where(df.loc[non_index_mask, "tradingValueSource"].isin(["KRX_STOCK", "KRX_ETF"])))],
    ["KRX exact 시가총액 coverage %", coverage(df.loc[non_index_mask, "marketCap"])],
    ["KRX 외국인 순매수금액 coverage %", coverage(df.loc[non_index_mask, "foreignNetBuyValue"])],
    ["KRX 기관 순매수금액 coverage %", coverage(df.loc[non_index_mask, "institutionNetBuyValue"])],
    ["Toss 외국인 순매수수량 coverage %", coverage(df.loc[non_index_mask, "foreignNetBuyVolume"])],
    ["Toss 외국인 보유량 coverage %", coverage(df.loc[non_index_mask, "foreignHoldingQuantity"])],
    ["Toss 개인 순매수수량 coverage %", coverage(df.loc[non_index_mask, "individualNetBuyVolume"])],
    ["Toss 기관 순매수수량 coverage %", coverage(df.loc[non_index_mask, "institutionNetBuyVolume"])],
    ["Toss 공매도 거래량 coverage %", coverage(df.loc[non_index_mask, "shortSellingVolume"])],
    ["Toss 대차잔고 coverage %", coverage(df.loc[non_index_mask, "lendingBalanceQuantity"])],
    ["Toss 신용융자잔고 coverage %", coverage(df.loc[non_index_mask, "marginLoanBalanceQuantity"])],
    ["Toss 프로그램 순매수수량 coverage %", coverage(df.loc[non_index_mask, "programNetBuyVolume"])],
    ["ETF NAV coverage %", coverage(df.loc[df["securityType"] == "ETF", "etfNav"])],
    ["ETF AUM coverage %", coverage(df.loc[df["securityType"] == "ETF", "etfNetAssetTotalAmount"])],
    ["오류 수", len(errors)],
]
summary = pd.DataFrame(summary_rows, columns=["항목", "결과"])

# 필드 사전
field_dictionary = pd.DataFrame([
    ["open/high/low/close/volume", "Toss /candles adjusted=true", "원천", "가격·기술지표"],
    ["tradingValue", "KRX", "원천(없으면 close×volume proxy)", "유동성/섹터 회전"],
    ["marketCap", "KRX", "원천", "시총필터/시총가중"],
    ["listedShares", "KRX", "원천", "발행/상장주식수"],
    ["foreignNetBuyValue", "KRX", "원천 KRW", "CloudTrend 외국인 수급"],
    ["institutionNetBuyValue", "KRX", "원천 KRW", "기관/섹터 수급"],
    ["individual/foreign/institution/*NetBuyVolume", "Toss investor-trading", "원천 주식수", "투자자 수급"],
    ["foreignHolding*", "Toss investor-trading", "원천", "외국인 실제 보유"],
    ["기관 7개 세부 NetBuyVolume", "Toss investor-trading", "원천 주식수", "기관 세부 수급"],
    ["shortSelling*", "Toss short-selling", "원천", "공매도"],
    ["lending*", "Toss securities-lending", "원천", "대차잔고"],
    ["marginLoan*/stockLoan*", "Toss credit-trades", "원천", "신용융자/신용대주"],
    ["program*", "Toss program-trades", "원천(KRX 거래만)", "프로그램매매"],
    ["etfNav/AUM/listedUnits/deviation/trackingError", "KRX ETF", "원천", "ETF 품질/자금흐름"],
], columns=["필드", "source", "quality", "용도"])

# 날짜 단위 분할 CSV 저장
unique_dates = sorted(df["date"].dropna().unique())
saved_paths = []
for i in range(0, len(unique_dates), CHUNK_DAYS):
    dates = unique_dates[i:i+CHUNK_DAYS]
    start_d = str(dates[0]).replace("-", "")
    end_d = str(dates[-1]).replace("-", "")
    out = df[df["date"].isin(dates)].copy()
    path = OUTPUT_ROOT / f"trendscore_input_{start_d}_{end_d}.csv"
    out.to_csv(path, index=False, encoding="utf-8-sig", na_rep="")
    saved_paths.append(path)

field_dictionary_path = OUTPUT_ROOT / "cloudtrend_integrated_field_dictionary.csv"
field_dictionary.to_csv(field_dictionary_path, index=False, encoding="utf-8-sig")

error_path = None
if errors:
    error_path = OUTPUT_ROOT / "cloudtrend_integrated_errors.csv"
    pd.DataFrame(errors).to_csv(error_path, index=False, encoding="utf-8-sig")

print("=" * 80)
print("CloudTrend Toss + KRX 통합 데이터 생성 완료")
print(f"Toss schema runtime version: {TOSS_SCHEMA_VERSION}")
print(f"행: {len(df):,} / symbols: {df['symbol'].nunique():,}")
print(f"CSV 분할 파일: {len(saved_paths):,}")
print("=" * 80)

display(summary)
display(field_dictionary)
display(df.head(20))

print("\n출처별 거래대금 행 수")
display(df.groupby("tradingValueSource", dropna=False).size().rename("rows").reset_index())

if errors:
    print("\n오류 단계별")
    display(
        pd.DataFrame(errors)
        .groupby("stage", dropna=False)
        .size()
        .rename("count")
        .reset_index()
        .sort_values("count", ascending=False)
    )

print("\n생성 파일")
for p in saved_paths:
    display(FileLink(str(p)))
display(FileLink(str(field_dictionary_path)))
if error_path:
    display(FileLink(str(error_path)))
